# GA Crossover Operators (Multi-Vehicle Routing)

Este notebook demonstra **4 operadores de crossover** usados na implementação do GA deste projeto:

1. **Order Crossover (OX)**
2. **Cycle Crossover (CX)**
3. **Partially Mapped Crossover (PMX)**
4. **Edge Recombination Crossover (ERX)**

Os operadores trabalham sobre a **permutação flat de destinos** (sem o `origin`) e retornam um filho válido no mesmo formato multi-veículo.

Cada célula de código abaixo cria dois pais de exemplo e mostra o resultado do crossover.

In [ ]:
#!pip install ipykernel
# Setup básico para os exemplos
import sys
from pathlib import Path

# Garantir que o root do repositório esteja no sys.path para importar o pacote src
# (o notebook fica em /concepts, então subimos um nível para o root do projeto)
sys.path.insert(0, str(Path("..").resolve()))

from src.infrastructure.genetic_algorithm.crossover import (
    OrderCrossoverStrategy,
    CycleCrossoverStrategy,
    PartiallyMappedCrossoverStrategy,
    EdgeRecombinationCrossoverStrategy,
)
from src.domain.models.geo_graph.route_node import RouteNode

# Origem comum usada em todos os veículos (não participa do crossover)
origin = RouteNode(name="Origin", node_id=0, coords=(0.0, 0.0))

# Destinos de exemplo (um único veículo com mais nós para ver melhor mistura no CX)
A = RouteNode(name="A", node_id=1, coords=(1.0, 0.0))
B = RouteNode(name="B", node_id=2, coords=(2.0, 0.0))
C = RouteNode(name="C", node_id=3, coords=(2.0, 1.0))
D = RouteNode(name="D", node_id=4, coords=(1.0, 1.0))
E = RouteNode(name="E", node_id=5, coords=(0.0, 1.0))
F = RouteNode(name="F", node_id=6, coords=(0.0, 0.0))
G = RouteNode(name="G", node_id=7, coords=(1.5, 1.5))
H = RouteNode(name="H", node_id=8, coords=(2.5, 0.5))

# Dois pais com ordens diferentes (1 veículo cada)
parent1 = [[origin, A, B, C, D, E, F, G, H]]
parent2 = [[origin, C, A, E, G, B, D, F, H]]

def names(individual):
    return [[node.name for node in route] for route in individual]

print("Parent 1:", names(parent1))
print("Parent 2:", names(parent2))

Parent 1: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Parent 2: [['Origin', 'C', 'A', 'E', 'G', 'B', 'D', 'F', 'H']]


## 1) Order Crossover (OX)

O **Order Crossover** é um operador simples e eficiente projetado para preservar a **ordem relativa** em vetores de permutação (como no Caixeiro Viajante - TSP). Ele funciona em 3 passos básicos:

1. **Slice (Corte):** Um trecho aleatório (sub-lista contígua) é escolhido do Pai 1 e copiado diretamente para a mesma posição no Filho.
2. **Coleta de Restantes:** Percorremos os genes do Pai 2 (da esquerda para a direita ou a partir do ponto de corte) filtrando os que **já foram copiados** no passo anterior.
3. **Preenchimento:** Colocamos os genes restantes do Pai 2 nos espaços vazios do Filho, garantindo que a sequência em que apareciam no Pai 2 seja respeitada.

**Vantagem Principal:** Ajuda a manter configurações úteis que não dependem da posição exata, mas da sequência em que os nós são visitados.

In [28]:
# A. Implementação "Bare-metal" para entender a lógica pura
def pure_order_crossover(p1, p2):
    length = len(p1)
    
    # 1. Definimos posições de corte fixas para o exemplo (em vez de aleatórias)
    start, end = 2, 5
    print(f"Slice do Pai 1 (índices {start} a {end-1}): {p1[start:end]}")
    
    # Preparamos o filho com espaços vazios
    child = [None] * length
    
    # Copiamos o slice do Pai 1
    for i in range(start, end):
        child[i] = p1[i]
        
    # 2. Pegamos os genes do Pai 2 que ainda não estão no filho
    remaining_genes = [gene for gene in p2 if gene not in child]
    print(f"Genes do Pai 2 restantes: {remaining_genes}")
    
    # 3. Preenchemos os espaços vazios do filho (antes e depois do corte)
    empty_positions = [i for i in range(length) if i < start or i >= end]
    for pos, gene in zip(empty_positions, remaining_genes):
        child[pos] = gene
        
    return child

sim_p1 = ["A", "B", "C", "D", "E", "F", "G", "H"]
sim_p2 = ["E", "A", "C", "H", "B", "D", "G", "F"]

print(f"Pai 1 (Simples): {sim_p1}")
print(f"Pai 2 (Simples): {sim_p2}")
sim_child = pure_order_crossover(sim_p1, sim_p2)
print(f"Filho (Simples): {sim_child}")

Pai 1 (Simples): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Pai 2 (Simples): ['E', 'A', 'C', 'H', 'B', 'D', 'G', 'F']
Slice do Pai 1 (índices 2 a 4): ['C', 'D', 'E']
Genes do Pai 2 restantes: ['A', 'H', 'B', 'G', 'F']
Filho (Simples): ['A', 'H', 'C', 'D', 'E', 'B', 'G', 'F']


---
### Uso com `OrderCrossoverStrategy` no repositório

A classe do projeto abstrai a preparação dos veículos, fatiando as classes de `RouteNode`, aplicando o crossover com cortes aleatórios e remontando a estrutura correta.

In [33]:
# B. Usando a classe de domínio do projeto
ox = OrderCrossoverStrategy()

# O crossover() roda na estrutura de multi-rotas com origem:
child_ox = ox.crossover(parent1, parent2)
print("Parent 1:", names(parent1))
print("Parent 2:", names(parent2))
print("Child (OrderCrossoverStrategy):", names(child_ox))

Parent 1: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Parent 2: [['Origin', 'C', 'A', 'E', 'G', 'B', 'D', 'F', 'H']]
Child (OrderCrossoverStrategy): [['Origin', 'C', 'A', 'E', 'B', 'D', 'F', 'G', 'H']]


## 2) Cycle Crossover (CX)

O **Cycle Crossover** se apoia totalmente nas **posições absolutas** dos cromossomos pais. Ele garante que todo destino no filho ocupe o mesmo índice em que ele estava no Pai 1 ou no Pai 2.

A "mágica" ocorre acompanhando ciclos através do mapa index-valor entre os dois pais:
1. Comece pelo índice 0. Analise o nó do `Pai 1` na posição 0. Reserve.
2. Veja qual nó o `Pai 2` possui nesse mesmo índice 0. Onde esse nó está no `Pai 1`?
3. Encontre o índice e preencha até o ciclo fechar (o nó do `Pai 2` for o nó original com que começamos).
4. Todo esse "ciclo" é garantido do Pai 1. Para os nós que sobraram sem atribuição, aplicamos o mesmo processo, mas tirando do Pai 2. E assim, alternamos em cada novo ciclo.

**Vantagem Principal:** Maximiza a herança das posições exatas dos nós. Se as posições da lista (slots em horário agendado, por exemplo) importam muito, CX é uma boa escolha.

In [35]:
# A. Implementação "Bare-metal"
def pure_cycle_crossover(p1, p2):
    length = len(p1)
    child = [None] * length
    unassigned = set(range(length))
    use_p1 = True
    
    cycle_num = 1
    while unassigned:
        start_index = min(unassigned)
        cycle_indexes = []
        current_index = start_index
        
        # Encontra o ciclo rastreando posições
        while current_index not in cycle_indexes:
            cycle_indexes.append(current_index)
            # Qual o valor nessa posição no P2?
            val_p2 = p2[current_index]
            # Qual o índice desse mesmo valor no P1?
            current_index = p1.index(val_p2)
            
        print(f"Encontrado Ciclo #{cycle_num} nos índices {cycle_indexes}. Usar P1? {use_p1}")
        cycle_num += 1
        
        # Preenche o filho
        source_parent = p1 if use_p1 else p2
        for i in cycle_indexes:
            child[i] = source_parent[i]
            unassigned.discard(i)
            
        # Alterna para o próximo ciclo vir do outro pai
        use_p1 = not use_p1
        
    return child

sim_p1_cx = ["A", "B", "C", "D", "E", "F", "G", "H"]
sim_p2_cx = ["C", "A", "E", "G", "B", "D", "F", "H"]

print(f"Pai 1: {sim_p1_cx}")
print(f"Pai 2: {sim_p2_cx}")
sim_child_cx = pure_cycle_crossover(sim_p1_cx, sim_p2_cx)
print(f"Filho: {sim_child_cx}")

Pai 1: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Pai 2: ['C', 'A', 'E', 'G', 'B', 'D', 'F', 'H']
Encontrado Ciclo #1 nos índices [0, 2, 4, 1]. Usar P1? True
Encontrado Ciclo #2 nos índices [3, 6, 5]. Usar P1? False
Encontrado Ciclo #3 nos índices [7]. Usar P1? True
Filho: ['A', 'B', 'C', 'G', 'E', 'D', 'F', 'H']


---
### Uso com `CycleCrossoverStrategy` no repositório
O repositório abstrai o achado lidando com instâncias de `RouteNode`, comparando usando a propriedade `node_id`.

In [36]:
# B. Usando a classe de domínio do projeto
cx = CycleCrossoverStrategy()
child_cx = cx.crossover(parent1, parent2)
print("Child (CycleCrossoverStrategy):", names(child_cx))

Child (CycleCrossoverStrategy): [['Origin', 'A', 'B', 'C', 'G', 'E', 'D', 'F', 'H']]


## 3) Partially Mapped Crossover (PMX)

O **Partially Mapped Crossover (PMX)** é possivelmente o mais conhecido operador genético de rotas focado em permutação por ser altamente estável. Ele balanceia a cópia de seções inteiras com a preservação de ordem do restante.

Assim como no OX, escolhemos um trecho do Pai 1 e copiamos. Porém a resolução do restante é totalmente diferente:
1. Ao invés de ignorar e copiar em ordem os itens faltantes, o PMX estabelece um **mapeamento reverso (mapping)** entre os itens que coincidiram no trecho cortado entre o Pai 1 e Pai 2.
2. Para preencher os índices fora do corte, tentamos manter o elemento do Pai 2, **a menos que** esse elemento já faça parte da herança direta do Pai 1. 
3. Se gerar duplicata, usamos o mapa construído no passo 1 para iterativamente pular pelos "espelhos" e achar o elemento legal disponível para preencher aquela lacuna.

**Vantagem Principal:** Preserva a estrutura absoluta do "miolo" e garante forte constância lógica herdada. Exatamente ideal para problemas como roteirização (TSP).

In [37]:
# A. Implementação "Bare-metal"
def pure_pmx_crossover(p1, p2):
    length = len(p1)
    child = [None] * length
    start, end = 2, 5  # indices do slice (corte fixado para o log)
    print(f"Trecho de corte do P1 índice {start} a {end-1}: {p1[start:end]}")
    
    # Copia o "middle slice" do Pai 1
    for i in range(start, end):
        child[i] = p1[i]
        
    print(f"Filho antes da resolução de conflitos: {child}")
    
    # Tenta copiar o "resto" do Pai 2. Onde houver conflito, mapear!
    for i in range(length):
        if i >= start and i < end:
            continue # pula o miolo
            
        candidate = p2[i]
        # mapeamento interativo até achar um slot seguro
        loop_c = 0
        while candidate in child:
            loop_c += 1
            # Qual foi o elemento que gerou o conflito (aquele q está no trecho herdado do P1)?
            idx_in_p1 = p1.index(candidate) 
            # Pegue quem estava no Pai 2 na mesma posição
            candidate = p2[idx_in_p1]
            if loop_c > length: # Segurança, não ocorre se lista é válida
                break 
                
        child[i] = candidate
        
    return child

print(f"Pai 1: {sim_p1}")
print(f"Pai 2: {sim_p2}")
sim_child_pmx = pure_pmx_crossover(sim_p1, sim_p2)
print(f"Filho: {sim_child_pmx}")

Pai 1: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Pai 2: ['E', 'A', 'C', 'H', 'B', 'D', 'G', 'F']
Trecho de corte do P1 índice 2 a 4: ['C', 'D', 'E']
Filho antes da resolução de conflitos: [None, None, 'C', 'D', 'E', None, None, None]
Filho: ['B', 'A', 'C', 'D', 'E', 'H', 'G', 'F']


---
### Uso com `PartiallyMappedCrossoverStrategy` no repositório

As operações reais de mapping entre os cromossomos podem ser observadas no `target_index` da classe nativa da infraestrutura.

In [43]:
# B. Usando a classe de domínio do projeto
pmx = PartiallyMappedCrossoverStrategy()
child_pmx = pmx.crossover(parent1, parent2)
print("Parent 1:", names(parent1))
print("Parent 2:", names(parent2))
print("Child (PartiallyMappedCrossoverStrategy):", names(child_pmx))

Parent 1: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Parent 2: [['Origin', 'C', 'A', 'E', 'G', 'B', 'D', 'F', 'H']]
Child (PartiallyMappedCrossoverStrategy): [['Origin', 'E', 'B', 'C', 'G', 'A', 'D', 'F', 'H']]


## 4) Edge Recombination Crossover (ERX)

O **Edge Recombination Crossover (ERX)** foi projetado especificamente para problemas como o TSP porque ele se preocupa exclusivamente em **preservar arestas (vizinhanças)** e não focar tão somente na ordenação pura. Seu objetivo é garantir que se nas rotas dos pais a cidade 'A' era visitada logo antes ou depois da cidade 'B', o filho deve imitar essa conexão, se possível.

A lógica é a mais diferente dos outros 3 métodos:
1. **Mapa de Adjacência:** Varremos o Pai 1 e Pai 2 fundindo uma lista de quais "vizinhos" cada nó possui em *qualquer* um dos pais (esquerda ou direita).
2. **Escolha Inicial:** Começamos o filho escolhendo aleatoriamente o primeiro nó. E este nó passa a ser o seu nó atual.
3. **Poda e Caminhada:**
   - Removemos as referências do seu "nó atual" de todo mapa para não voltar para ele.
   - Analisamos todos os vizinhos disponíveis sob a chave do seu nó atual.
   - De todos os vizinhos possíveis, escolhemos aquele que tem o **MENOR** número de vizinhos restantes não vistados. (Heurística que força a "salvar" sub-rotas críticas que poderiam ser abandonadas).
   - Se ocorrer um empate de mínimo de arestas, o nó será aleatório entre os empatados.
   - Se o seu nó não tem mais vizinhos legais, escolha um restante na sorte. Repita até zerar.

**Vantagem Principal:** O mais robusto em preservar trechos sub-ótimos exatos. É computacionalmente sensível, mas produz resultados estruturais superiores em problemas baseados em grafos.

In [44]:
import random
# A. Implementação "Bare-metal"
def pure_erx_crossover(p1, p2):
    # 1. Cria mapa misturando arestas adjacentes
    def map_edges(seq):
        adj = {item: set() for item in seq}
        length = len(seq)
        for i, item in enumerate(seq):
            if i > 0: adj[item].add(seq[i-1])
            if i < length - 1: adj[item].add(seq[i+1])
        return adj
        
    edge_map = map_edges(p1)
    for node, neighbors in map_edges(p2).items():
        edge_map[node].update(neighbors)
        
    print("Edge map unificado:", {k: list(v) for k, v in edge_map.items()})
    
    # 2. Executa a caminhada
    remaining = set(p1)
    current = p1[0]  # em producao iniciar randomicamente
    child = []
    
    while remaining:
        child.append(current)
        remaining.discard(current)
        
        # Poda referências do atual de todos os vizinhos
        for neighbors in edge_map.values():
            neighbors.discard(current)
            
        if not remaining:
            break
            
        # Pega os vizinhos validos do nó atual
        valid_neighbors = [n for n in edge_map.get(current, set()) if n in remaining]
        
        # 3. Das opções, escolha o que tem MENOS pontes futuras a perder
        if valid_neighbors:
            min_degree = min(len(edge_map[n]) for n in valid_neighbors)
            best_candidates = [n for n in valid_neighbors if len(edge_map[n]) == min_degree]
            current = best_candidates[0] # pegue o primeiro pro log, em producao é random
        else:
            # Falha de vizinhança: fallback para random
            current = list(remaining)[0]
    return child

print(f"Pai 1: {sim_p1}")
print(f"Pai 2: {sim_p2}")
sim_child_erx = pure_erx_crossover(sim_p1, sim_p2)
print(f"Filho: {sim_child_erx}")

Pai 1: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
Pai 2: ['E', 'A', 'C', 'H', 'B', 'D', 'G', 'F']
Edge map unificado: {'A': ['B', 'C', 'E'], 'B': ['D', 'C', 'A', 'H'], 'C': ['D', 'B', 'A', 'H'], 'D': ['B', 'C', 'G', 'E'], 'E': ['D', 'A', 'F'], 'F': ['G', 'E'], 'G': ['D', 'H', 'F'], 'H': ['C', 'B', 'G']}
Filho: ['A', 'E', 'F', 'G', 'D', 'B', 'C', 'H']


---
### Uso com `EdgeRecombinationCrossoverStrategy` no repositório

A classe em `src/infrastructure/genetic_algorithm/crossover/edge_recombination_crossover_strategy.py` aplica essa idêntica lógica resolvendo pelo ID base `RouteNode.node_id`.

In [47]:
# B. Usando a classe de domínio do projeto
erx = EdgeRecombinationCrossoverStrategy()
child_erx = erx.crossover(parent1, parent2)
print("Child (EdgeRecombinationCrossoverStrategy):", names(child_erx))

Child (EdgeRecombinationCrossoverStrategy): [['Origin', 'F', 'H', 'G', 'E', 'A', 'C', 'D', 'B']]
